# Claude API Learning Journey

## Step 1 — First API Request
Install dependencies, load API key from `.env`, initialise the Anthropic client, and make the first `client.messages.create()` call.

## Step 2 — Multi-Turn Conversations
Claude is stateless — maintain a `messages` list and send the full history with every request. Helper functions: `add_user_message()`, `add_assistant_message()`, `chat()`.

## Step 3 — Chat Exercise: Continuous Chat Loop
`while True` loop that accepts user input, sends the full conversation history to Claude, appends the response, and repeats — creating a stateful terminal chat session.

## Step 4 — System Prompts
Shape Claude's role and behaviour with a `system` parameter. Updated `chat()` to accept an optional `system` arg (conditionally included — API rejects `system=None`).

## Step 5 — System Prompts Exercise
Applied a one-sentence system prompt `"You are a Python engineer who writes very concise code"` — Claude returned a 1-line solution vs. a verbose multi-function response without the prompt.

## Step 6 — Temperature
`temperature` (0.0–1.0) controls token selection randomness. Low → deterministic/factual. High → creative/varied. Added as an optional param to `chat()` defaulting to `1.0`. Verified with movie idea generation — temperature 0 produced near-identical responses across 3 runs, temperature 1 produced distinct ideas each time.

## Step 7 — Response Streaming
Stream Claude's response word by word using `stream=True` or `client.messages.stream()`. Three patterns: raw events, simplified `text_stream`, and `get_final_message()` for the assembled result after streaming.

## Step 8 — Structured Data
Force clean structured output (JSON, code, CSV) without markdown wrappers or explanatory prose. Course technique: assistant message prefill + stop sequences — not supported on Claude 4. Workaround: system prompt instructing raw output directly.

## Step 9 — Structured Data Exercise
Applied the system prompt workaround to generate raw AWS CLI bash commands. Course prefill approach (` ```bash ` + stop sequence) fails on Claude 4 — replaced with `system="Output raw bash commands only..."` for compatible clean output.

In [23]:
# Install Dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [24]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [25]:
# Create an API client
from anthropic import Anthropic 
client = Anthropic()
model = "claude-sonnet-4-6"

In [64]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
        
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    return message.content[0].text

In [74]:
messages = []

add_user_message(messages, "Generate three different sample AWS CLI commands. Each should be very short")

# Claude 4 doesn't support assistant prefill — use system prompt to enforce raw bash output
text = chat(
    messages,
    system="Output raw bash commands only, one per line, no numbering, no markdown, no explanation."
)
print(text)

aws s3 ls
aws ec2 describe-instances
aws iam list-users
